In [5]:
!pip install --upgrade -qq  langchain langchain-core langchain-community langchain-experimental langchain_openai langchain-google-genai langchain-ollama langchain-anthropic langchain-tavily
!pip install praw yfinance asknews



  Using cached praw-7.8.1-py3-none-any.whl.metadata (9.4 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.8 MB/s eta 0:00:00
  Created wheel for crontab: filename=crontab-1.0.5-py3-none-any.whl size=20815 sha256=c5efa2e2aa9f145375900e32c967276ef41f50f4128022496192663c8bcb8efa
  Stored in directory: /root/.cache/pip/wheels/a9/f3/f2/bee2e146d7d11640b35cbcdd268d694565e341002b89cac048
Successfully built crontab


In [6]:
%%bash

# pip install colab-xterm
sudo apt install pciutils lshw zstd
curl -fsSL https://ollama.com/install.sh | sh

Reading package lists...
Building dependency tree...
Reading state information...
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2build1).
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.




>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# %%bash

# load_ext colabxterm
# xterm

# ollama serve

In [7]:
import os
import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"],
                           stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL)

time.sleep(10)

Starting Ollama server...


In [8]:
!ollama pull qwen3:latest

os.environ["DEFAULT_NUM_CTX"] = "16384"

In [9]:
import os
from dotenv import load_dotenv
import json
import operator
import numpy as np
from pydantic import BaseModel, Field
import json
import praw
import tweepy
import traceback
import yfinance as yf
import sqlite3
import gradio as gr
import uuid
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, List, Tuple, Dict, Any, Literal, Union
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_community.utilities.polygon import PolygonAPIWrapper
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_tavily import TavilySearch
from langchain_community.retrievers import AskNewsRetriever
from langchain_community.document_loaders import (
    RedditPostsLoader,
    TwitterTweetLoader,
)
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langsmith import Client, traceable

In [10]:
# Load environment variables

load_dotenv()

True

In [11]:
# ===========================
# Config
# ===========================

class Config:
    """Configuration from environment variables"""

    LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
    LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "stock-trading-agent")
    LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING_V2", "true")

    TWITTER_API_KEY = os.getenv("TWITTER_API_KEY")
    TWITTER_API_SECRET = os.getenv("TWITTER_API_SECRET")
    TWITTER_ACCESS_TOKEN = os.getenv("TWITTER_ACCESS_TOKEN")
    TWITTER_ACCESS_SECRET = os.getenv("TWITTER_ACCESS_SECRET")
    TWITTER_BEARER_TOKEN = os.getenv("TWITTER_BEARER_TOKEN")

    REDDIT_CLIENT_ID = os.getenv("REDDIT_CLIENT_ID")
    REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET")
    REDDIT_USER_AGENT = os.getenv("REDDIT_USER_AGENT", "StockTradingAgent/1.0")

    POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")
    TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
    ASKNEWS_CLIENT_ID = os.getenv("ASKNEWS_CLIENT_ID")
    ASKNEWS_CLIENT_SECRET = os.getenv("ASKNEWS_CLIENT_SECRET")

    MAX_TWEETS = int(os.getenv("MAX_TWEETS", "50"))
    MAX_REDDIT_POSTS = int(os.getenv("MAX_REDDIT_POSTS", "50"))
    SOCIAL_LOOKBACK_DAYS = int(os.getenv("SOCIAL_LOOKBACK_DAYS", "7"))

    MODEL_TEMPERATURE = float(os.getenv("MODEL_TEMPERATURE", "0"))

    DEFAULT_RISK_TOLERANCE = os.getenv("DEFAULT_RISK_TOLERANCE", "medium")
    MAX_POSITION_SIZE = float(os.getenv("MAX_POSITION_SIZE", "0.15"))
    MIN_CONFIDENCE_THRESHOLD = int(os.getenv("MIN_CONFIDENCE_THRESHOLD", "60"))

    LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").lower()  # Options: anthropic, ollama, openai, gemini

    ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
    ANTHROPIC_MODEL_NAME = os.getenv("ANTHROPIC_MODEL_NAME", "claude-haiku-4-5-20251001")

    OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:latest")
    OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4-turbo-preview")

    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    GOOGLE_MODEL = os.getenv("GOOGLE_MODEL", "gemini-3-pro-preview")

    DB_PATH = os.getenv("DB_PATH", "trading_history.db")

In [12]:
# ===========================
# Risk Assessment
# ===========================

class RiskAssessment(BaseModel):
    """Structured risk assessment output"""
    risk_level: Literal["low", "medium", "high", "very_high"] = Field(
        description="Overall risk level for the investment"
    )

    risk_score: int = Field(
        description="Risk score from 0-100, where 0 is lowest risk",
        ge=0,
        le=100
    )

    recommended_position_size: float = Field(
        description="Recommended position size as decimal (0-1)",
        ge=0,
        le=100
    )

    key_risks: List[str] = Field(
        description="List of key risks identified"
    )

    risk_mitigation_strategies: List[str] = Field(
        description="Recommended strategies to mitigate risks"
    )

    volatility_assessment: Literal["low", "medium", "high", "extreme"] = Field(
        description="Assessment of price volatility"
    )

In [13]:
# ===========================
# Indicator Signal
# ===========================

class IndicatorSignal(BaseModel):
    """Signal from a single indicator"""
    indicator_name: str = Field(
        description="Name of the indicator"
    )

    signal: Literal["BUY", "SELL", "HOLD"] = Field(
        description="Trading signal"
    )

    confidence: int = Field(
        description="Confidence level 0-100",
        ge=0,
        le=100
    )

    current_value: float = Field(
        description="Current indicator value"
    )

    reasoning: str = Field(
        description="Explanation for the signal"
    )

In [14]:
# ===========================
# Indicators Voting Result
# ===========================

class VotingResult(BaseModel):
    """Combined voting result from all indicators"""
    final_signal: Literal["BUY", "SELL", "HOLD"] = Field(
        description="Final consensus signal"
    )

    confidence: int = Field(
        description="Overall confidence 0-100", ge=0, le=100
    )

    buy_votes: int = Field(
        description="Number of BUY votes"
    )

    sell_votes: int = Field(
        description="Number of SELL votes"
    )

    hold_votes: int = Field(
        description="Number of HOLD votes"
    )

    indicators: List[IndicatorSignal] = Field(
        description="Individual indicator signals"
    )

In [15]:
# ===========================
# State Definitions
# ===========================

class AgentState(TypedDict):
    """Enhanced state for multi-agent trading system"""
    ticker: str
    messages: Annotated[List[Dict], operator.add]

    market_data: Dict[str, Any]
    technical_analysis: Dict[str, Any]
    fundamental_analysis: Dict[str, Any]
    sentiment_analysis: Dict[str, Any]
    risk_analysis: Dict[str, Any]

    agent_outputs: Annotated[List[Dict], operator.add]

    recommendation: Dict[str, Any]

    next_agent: str

In [16]:
# ===========================
# Technical Indicators
# ===========================

class TechnicalIndicators:
    """Technical indicators with voting system - provides confirmation signals"""

    @staticmethod
    def calculate_rsi(prices: List[float], period: int = 14) -> Dict[str, Any]:
        """Calculate RSI (Relative Strength Index)"""
        if len(prices) < period + 1:
            return {"error": "Insufficient data", "signal": "HOLD"}

        deltas = np.diff(prices)
        gains = np.where(deltas > 0, deltas, 0)
        losses = np.where(deltas < 0, -deltas, 0)

        avg_gain = np.mean(gains[-period:])
        avg_loss = np.mean(losses[-period:])

        if avg_loss == 0:
            rsi = 100
        else:
            rs = avg_gain / avg_loss
            rsi = 100 - (100 / (1 + rs))

        if rsi > 70:
            signal = "SELL"
            confidence = min(int((rsi - 70) * 3), 100)
            reasoning = f"RSI at {rsi:.2f} indicates overbought conditions"
        elif rsi < 30:
            signal = "BUY"
            confidence = min(int((30 - rsi) * 3), 100)
            reasoning = f"RSI at {rsi:.2f} indicates oversold conditions"
        else:
            signal = "HOLD"
            confidence = int(abs(50 - rsi))
            reasoning = f"RSI at {rsi:.2f} is in neutral territory"

        return {
            "rsi": round(rsi, 2),
            "signal": signal,
            "confidence": confidence,
            "reasoning": reasoning,
            "overbought": rsi > 70,
            "oversold": rsi < 30
        }

    @staticmethod
    def calculate_alligator(highs: List[float], lows: List[float], closes: List[float]) -> Dict[str, Any]:
        """Calculate Alligator Indicator (Bill Williams)"""
        if len(closes) < 21:
            return {"error": "Insufficient data", "signal": "HOLD"}

        median_prices = [(h + l) / 2 for h, l in zip(highs, lows)]

        def smma(data: List[float], period: int) -> List[float]:
            """Smoothed Moving Average"""
            result = []
            sma = np.mean(data[:period])
            result.append(sma)

            for i in range(period, len(data)):
                smma_val = (result[-1] * (period - 1) + data[i]) / period
                result.append(smma_val)

            return result

        jaw = smma(median_prices, 13)
        teeth = smma(median_prices, 8)
        lips = smma(median_prices, 5)

        current_price = closes[-1]
        jaw_current = jaw[-1] if len(jaw) > 0 else current_price
        teeth_current = teeth[-1] if len(teeth) > 0 else current_price
        lips_current = lips[-1] if len(lips) > 0 else current_price

        if lips_current > teeth_current > jaw_current:
            if current_price > lips_current:
                signal = "BUY"
                confidence = 80
                reasoning = "Alligator shows strong bullish trend - all lines aligned upward"
            else:
                signal = "HOLD"
                confidence = 50
                reasoning = "Alligator bullish but price below lips - waiting for confirmation"
        elif lips_current < teeth_current < jaw_current:
            if current_price < lips_current:
                signal = "SELL"
                confidence = 80
                reasoning = "Alligator shows strong bearish trend - all lines aligned downward"
            else:
                signal = "HOLD"
                confidence = 50
                reasoning = "Alligator bearish but price above lips - waiting for confirmation"
        else:
            signal = "HOLD"
            confidence = 30
            reasoning = "Alligator lines intertwined - market is ranging"

        return {
            "jaw": round(jaw_current, 2),
            "teeth": round(teeth_current, 2),
            "lips": round(lips_current, 2),
            "current_price": round(current_price, 2),
            "signal": signal,
            "confidence": confidence,
            "reasoning": reasoning,
            "trend": "bullish" if lips_current > teeth_current > jaw_current else "bearish" if lips_current < teeth_current < jaw_current else "ranging"
        }

    @staticmethod
    def calculate_squeeze_momentum(highs: List[float], lows: List[float], closes: List[float],
                                   bb_length: int = 20, bb_mult: float = 2.0,
                                   kc_length: int = 20, kc_mult: float = 1.5) -> Dict[str, Any]:
        """Calculate Squeeze Momentum Indicator (LazyBear)"""
        if len(closes) < max(bb_length, kc_length) + 1:
            return {"error": "Insufficient data", "signal": "HOLD"}

        source = np.array(closes)
        high = np.array(highs)
        low = np.array(lows)

        basis = np.mean(source[-bb_length:])
        dev = bb_mult * np.std(source[-bb_length:])
        upper_bb = basis + dev
        lower_bb = basis - dev

        ma = np.mean(source[-kc_length:])

        tr_list = []
        for i in range(-kc_length, 0):
            if i == -kc_length:
                tr = high[i] - low[i]
            else:
                tr = max(high[i] - low[i], abs(high[i] - source[i-1]), abs(low[i] - source[i-1]))
            tr_list.append(tr)

        range_ma = np.mean(tr_list)
        upper_kc = ma + range_ma * kc_mult
        lower_kc = ma - range_ma * kc_mult

        sqz_on = (lower_bb > lower_kc) and (upper_bb < upper_kc)
        sqz_off = (lower_bb < lower_kc) and (upper_bb > upper_kc)
        no_sqz = not sqz_on and not sqz_off

        highest_high = np.max(high[-kc_length:])
        lowest_low = np.min(low[-kc_length:])
        mid_line = (highest_high + lowest_low) / 2
        sma_close = np.mean(source[-kc_length:])
        avg_val = (mid_line + sma_close) / 2

        data = source[-kc_length:] - avg_val
        x = np.arange(len(data))

        n = len(data)
        sum_x = np.sum(x)
        sum_y = np.sum(data)
        sum_xy = np.sum(x * data)
        sum_x2 = np.sum(x * x)

        slope = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x * sum_x)
        intercept = (sum_y - slope * sum_x) / n
        val = slope * (n - 1) + intercept

        if len(source) > kc_length + 1:
            prev_data = source[-(kc_length+1):-1] - avg_val
            prev_slope = (n * np.sum(x * prev_data) - sum_x * np.sum(prev_data)) / (n * sum_x2 - sum_x * sum_x)
            prev_intercept = (np.sum(prev_data) - prev_slope * sum_x) / n
            prev_val = prev_slope * (n - 1) + prev_intercept
        else:
            prev_val = 0

        momentum_increasing = val > prev_val
        momentum_positive = val > 0

        if sqz_on:
            if momentum_positive and momentum_increasing:
                signal = "BUY"
                confidence = 70
                reasoning = "Squeeze ON - potential bullish breakout"
            elif not momentum_positive and not momentum_increasing:
                signal = "SELL"
                confidence = 70
                reasoning = "Squeeze ON - potential bearish breakout"
            else:
                signal = "HOLD"
                confidence = 60
                reasoning = "Squeeze ON - consolidation, wait for direction"
        elif sqz_off:
            if momentum_positive and momentum_increasing:
                signal = "BUY"
                confidence = 90
                reasoning = "Squeeze fired OFF - bullish breakout confirmed"
            elif not momentum_positive and not momentum_increasing:
                signal = "SELL"
                confidence = 90
                reasoning = "Squeeze fired OFF - bearish breakout confirmed"
            else:
                signal = "HOLD"
                confidence = 40
                reasoning = "Squeeze OFF but momentum mixed"
        else:
            if momentum_positive and momentum_increasing:
                signal = "BUY"
                confidence = 60
                reasoning = "Positive increasing momentum"
            elif not momentum_positive and not momentum_increasing:
                signal = "SELL"
                confidence = 60
                reasoning = "Negative decreasing momentum"
            else:
                signal = "HOLD"
                confidence = 50
                reasoning = "No squeeze and mixed momentum"

        return {
            "squeeze_on": sqz_on,
            "squeeze_off": sqz_off,
            "momentum_value": round(val, 4),
            "momentum_direction": "positive" if momentum_positive else "negative",
            "momentum_trend": "increasing" if momentum_increasing else "decreasing",
            "signal": signal,
            "confidence": confidence,
            "reasoning": reasoning
        }

    @staticmethod
    def voting_system(rsi_result: Dict, alligator_result: Dict, squeeze_result: Dict) -> VotingResult:
        """Combine signals from all three indicators"""
        indicators = []

        if "error" not in rsi_result:
            indicators.append(IndicatorSignal(
                indicator_name="RSI",
                signal=rsi_result["signal"],
                confidence=rsi_result["confidence"],
                current_value=rsi_result["rsi"],
                reasoning=rsi_result["reasoning"]
            ))

        if "error" not in alligator_result:
            indicators.append(IndicatorSignal(
                indicator_name="Alligator",
                signal=alligator_result["signal"],
                confidence=alligator_result["confidence"],
                current_value=alligator_result["lips"],
                reasoning=alligator_result["reasoning"]
            ))

        if "error" not in squeeze_result:
            indicators.append(IndicatorSignal(
                indicator_name="Squeeze Momentum",
                signal=squeeze_result["signal"],
                confidence=squeeze_result["confidence"],
                current_value=squeeze_result["momentum_value"],
                reasoning=squeeze_result["reasoning"]
            ))

        buy_votes = sum(1 for ind in indicators if ind.signal == "BUY")
        sell_votes = sum(1 for ind in indicators if ind.signal == "SELL")
        hold_votes = sum(1 for ind in indicators if ind.signal == "HOLD")

        if buy_votes > sell_votes and buy_votes > hold_votes:
            final_signal = "BUY"
        elif sell_votes > buy_votes and sell_votes > hold_votes:
            final_signal = "SELL"
        else:
            final_signal = "HOLD"

        if len(indicators) > 0:
            total_confidence = sum(ind.confidence for ind in indicators if ind.signal == final_signal)
            count = sum(1 for ind in indicators if ind.signal == final_signal)
            confidence = int(total_confidence / count) if count > 0 else 50
        else:
            confidence = 50

        return VotingResult(
            final_signal=final_signal,
            confidence=confidence,
            buy_votes=buy_votes,
            sell_votes=sell_votes,
            hold_votes=hold_votes,
            indicators=indicators
        )

In [17]:
# ===========================
# Json type converter
# ===========================

def convert_numpy_types(obj):
    """Recursively convert NumPy types to Python native types for JSON serialization"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, dict):
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(item) for item in obj]
    else:
        return obj

In [18]:
# ===========================
# Tools Manager
# ===========================

class ToolsManager:
    """Manages all trading tools including social media and technical indicators"""

    def __init__(self):
        self.polygon = None
        if Config.POLYGON_API_KEY:
            self.polygon = PolygonAPIWrapper(polygon_api_key=Config.POLYGON_API_KEY)
        self.tech_indicators = TechnicalIndicators()

    def _get_yfinance_data(self, ticker: str, period: str = "1y"):
        """Get historical data from yfinance"""
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period=period)

            if hist.empty:
                return None

            results = []
            for index, row in hist.iterrows():
                results.append({
                    'c': row['Close'],
                    'h': row['High'],
                    'l': row['Low'],
                    'o': row['Open'],
                    'v': row['Volume'],
                    't': int(index.timestamp() * 1000)
                })

            return results

        except Exception as e:
            print(f"yfinance error: {e}")
            return None

    def get_market_data_tools(self) -> List:
        """Tools for market data with technical indicators - yfinance only"""
        tools = []

        @tool
        def technical_indicator_voting(ticker: str) -> str:
            """Calculate RSI, Alligator, Squeeze Momentum with voting system using yfinance"""
            try:
                end_date = datetime.now().strftime("%Y-%m-%d")
                start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

                print(f"📊 Fetching data for {ticker} from yfinance...")
                aggregates = self._get_yfinance_data(ticker, period="1y")
                data_source = "yfinance"

                if not aggregates or len(aggregates) < 50:
                    return json.dumps({
                        "error": "Insufficient data from yfinance",
                        "ticker": ticker,
                        "data_points": len(aggregates) if aggregates else 0,
                        "message": f"Could not gather enough historical data for {ticker}. Please verify the ticker symbol."
                    })

                print(f"✅ Using yfinance data: {len(aggregates)} days")

                closes = [float(agg.get('c', 0)) for agg in aggregates]
                highs = [float(agg.get('h', 0)) for agg in aggregates]
                lows = [float(agg.get('l', 0)) for agg in aggregates]

                rsi_result = self.tech_indicators.calculate_rsi(closes)
                alligator_result = self.tech_indicators.calculate_alligator(highs, lows, closes)
                squeeze_result = self.tech_indicators.calculate_squeeze_momentum(highs, lows, closes)

                voting_result = self.tech_indicators.voting_system(
                    rsi_result, alligator_result, squeeze_result
                )

                result_data = {
                    "ticker": ticker,
                    "current_price": round(float(closes[-1]), 2),
                    "data_source": data_source,
                    "data_points": len(aggregates),
                    "date_range": {
                        "start": start_date,
                        "end": end_date
                    },
                    "indicators": {
                        "rsi": convert_numpy_types(rsi_result),
                        "alligator": convert_numpy_types(alligator_result),
                        "squeeze_momentum": convert_numpy_types(squeeze_result)
                    },
                    "voting_result": convert_numpy_types(voting_result.model_dump())
                }

                return json.dumps(result_data, indent=2)

            except Exception as e:
                error_details = traceback.format_exc()
                print(f"ERROR in technical_indicator_voting: {error_details}")
                return json.dumps({
                    "error": str(e),
                    "ticker": ticker,
                    "note": "Technical analysis failed - continuing with other analyses",
                    "traceback": error_details[:500]
                })

        tools.append(technical_indicator_voting)
        return tools

    def get_sentiment_tools(self) -> List:
        """Tools for sentiment analysis including social media"""
        tools = []

        tools.append(YahooFinanceNewsTool())

        if Config.TAVILY_API_KEY:
            tavily_tool = TavilySearch(
                api_key=Config.TAVILY_API_KEY,
                max_results=5,
                search_depth="advanced",
                include_answer=True
            )

            @tool
            def web_news_search(query: str) -> str:
                """Search recent news and articles about stocks and market trends"""
                return tavily_tool.invoke(query)

            tools.append(web_news_search)

        if Config.ASKNEWS_CLIENT_ID and Config.ASKNEWS_CLIENT_SECRET:
            @tool
            def asknews_search(query: str) -> str:
                """Search recent news using AskNews - comprehensive news coverage with AI-powered analysis"""
                try:
                    retriever = AskNewsRetriever(
                        client_id=Config.ASKNEWS_CLIENT_ID,
                        client_secret=Config.ASKNEWS_CLIENT_SECRET,
                        k=5
                    )

                    documents = retriever.invoke(query)

                    news_items = []
                    for doc in documents:
                        news_items.append({
                            "content": doc.page_content[:500],
                            "source": doc.metadata.get('source', 'N/A'),
                            "title": doc.metadata.get('title', 'N/A'),
                            "url": doc.metadata.get('url', 'N/A'),
                            "published_at": doc.metadata.get('published_at', 'N/A')
                        })

                    return json.dumps({
                        "platform": "asknews",
                        "query": query,
                        "total_articles": len(news_items),
                        "articles": news_items
                    }, indent=2)

                except Exception as e:
                    return json.dumps({"error": str(e), "platform": "asknews"})

            tools.append(asknews_search)

        if Config.TWITTER_BEARER_TOKEN:
            @tool
            def twitter_sentiment(query: str) -> str:
                """Search Twitter/X for recent stock mentions"""
                try:
                    all_tweets = []

                    try:
                        finance_accounts = ["@federalreserve", "@business", "@WSJ", "@CNBC", "@Bloomberg"]

                        for account in finance_accounts:
                            try:
                                loader = TwitterTweetLoader(
                                    auth_handler=tweepy.OAuth2BearerHandler(Config.TWITTER_BEARER_TOKEN),
                                    twitter_users=[account.replace("@", "")],
                                    number_tweets=10
                                )

                                documents = loader.load()

                                for doc in documents:
                                    if query.upper() in doc.page_content.upper() or f"${query.upper()}" in doc.page_content:
                                        tweet_data = {
                                            "text": doc.page_content,
                                            "source": account,
                                            "metadata": doc.metadata
                                        }
                                        all_tweets.append(tweet_data)
                            except Exception as loader_error:
                                continue

                    except Exception as langchain_error:
                        client = tweepy.Client(bearer_token=Config.TWITTER_BEARER_TOKEN)
                        search_terms = [f"${query}", f"{query} stock"]

                        for search_term in search_terms:
                            search_query = f"{search_term} -is:retweet lang:en"

                            response = client.search_recent_tweets(
                                query=search_query,
                                max_results=min(Config.MAX_TWEETS // len(search_terms), 100),
                                tweet_fields=['created_at', 'public_metrics']
                            )

                            if response.data:
                                for tweet in response.data:
                                    metrics = tweet.public_metrics
                                    engagement = (
                                        metrics.get('like_count', 0) +
                                        metrics.get('retweet_count', 0) * 2 +
                                        metrics.get('reply_count', 0)
                                    )

                                    all_tweets.append({
                                        "text": tweet.text,
                                        "created_at": str(tweet.created_at),
                                        "engagement_score": engagement,
                                        "metrics": metrics
                                    })

                    if not all_tweets:
                        return json.dumps({"message": "No tweets found", "count": 0, "platform": "twitter"})

                    all_tweets.sort(key=lambda x: x.get('engagement_score', 0), reverse=True)
                    total_engagement = sum(t.get('engagement_score', 0) for t in all_tweets)

                    return json.dumps({
                        "platform": "twitter",
                        "query": query,
                        "total_tweets": len(all_tweets),
                        "total_engagement": total_engagement,
                        "avg_engagement": total_engagement / len(all_tweets) if all_tweets and total_engagement > 0 else 0,
                        "top_tweets": all_tweets[:15]
                    }, indent=2)

                except Exception as e:
                    return json.dumps({"error": str(e), "platform": "twitter"})

            tools.append(twitter_sentiment)

        if Config.REDDIT_CLIENT_ID and Config.REDDIT_CLIENT_SECRET:
            @tool
            def reddit_sentiment(query: str) -> str:
                """Search Reddit communities for stock discussions"""
                try:
                    subreddits = ['wallstreetbets', 'stocks', 'investing', 'stockmarket', 'options', 'daytrading']
                    all_posts = []

                    for subreddit_name in subreddits:
                        try:
                            loader = RedditPostsLoader(
                                client_id=Config.REDDIT_CLIENT_ID,
                                client_secret=Config.REDDIT_CLIENT_SECRET,
                                user_agent=Config.REDDIT_USER_AGENT,
                                search_queries=[query],
                                subreddit=subreddit_name,
                                mode="search",
                                categories=["hot", "new"],
                                number_posts=15,
                                time_filter="week"
                            )

                            documents = loader.load()

                            for doc in documents:
                                metadata = doc.metadata

                                post_data = {
                                    "subreddit": subreddit_name,
                                    "title": metadata.get("title", ""),
                                    "text": doc.page_content[:500],
                                    "score": metadata.get("score", 0),
                                    "upvote_ratio": metadata.get("upvote_ratio", 0),
                                    "num_comments": metadata.get("num_comments", 0),
                                    "created_utc": metadata.get("created_utc", ""),
                                    "url": metadata.get("post_url", ""),
                                    "author": metadata.get("author", ""),
                                    "engagement_score": metadata.get("score", 0) + (metadata.get("num_comments", 0) * 2)
                                }

                                all_posts.append(post_data)

                        except Exception as loader_error:
                            try:
                                reddit = praw.Reddit(
                                    client_id=Config.REDDIT_CLIENT_ID,
                                    client_secret=Config.REDDIT_CLIENT_SECRET,
                                    user_agent=Config.REDDIT_USER_AGENT
                                )

                                subreddit = reddit.subreddit(subreddit_name)
                                for post in subreddit.search(query, time_filter='week', limit=15):
                                    all_posts.append({
                                        "subreddit": subreddit_name,
                                        "title": post.title,
                                        "text": post.selftext[:500] if post.selftext else "",
                                        "score": post.score,
                                        "upvote_ratio": post.upvote_ratio,
                                        "num_comments": post.num_comments,
                                        "created_utc": datetime.fromtimestamp(post.created_utc).isoformat(),
                                        "engagement_score": post.score + (post.num_comments * 2)
                                    })
                            except:
                                continue

                    if not all_posts:
                        return json.dumps({"message": "No Reddit posts found", "count": 0, "platform": "reddit"})

                    all_posts.sort(key=lambda x: x['engagement_score'], reverse=True)

                    avg_score = sum(p['score'] for p in all_posts) / len(all_posts)
                    avg_ratio = sum(p['upvote_ratio'] for p in all_posts) / len(all_posts)
                    total_comments = sum(p['num_comments'] for p in all_posts)

                    return json.dumps({
                        "platform": "reddit",
                        "query": query,
                        "total_posts": len(all_posts),
                        "avg_score": round(avg_score, 2),
                        "avg_upvote_ratio": round(avg_ratio, 2),
                        "total_comments": total_comments,
                        "subreddits_searched": subreddits,
                        "sentiment_indicator": "bullish" if avg_ratio > 0.7 and avg_score > 50 else "bearish" if avg_ratio < 0.5 else "neutral",
                        "top_posts": all_posts[:15]
                    }, indent=2)

                except Exception as e:
                    return json.dumps({"error": str(e), "platform": "reddit"})

            tools.append(reddit_sentiment)

        return tools

    def get_fundamental_tools(self) -> List:
        """Tools for fundamental analysis using both Polygon (for news) and yfinance (for financials)"""

        tools = []

        if self.polygon:
            @tool
            def polygon_company_news(ticker: str) -> str:
                """Get company news and updates from Polygon API"""
                try:
                    details = self.polygon.get_ticker_news(ticker)
                    return json.dumps(details, indent=2)
                except Exception as e:
                    return json.dumps({
                        "error": str(e),
                        "note": "Polygon news retrieval failed"
                    })

            tools.append(polygon_company_news)

        @tool
        def company_financials(ticker: str) -> str:
            """Get detailed company financial metrics and ratios using yfinance"""
            try:
                stock = yf.Ticker(ticker)
                info = stock.info

                fundamentals = {
                    "company_name": info.get('longName', 'N/A'),
                    "sector": info.get('sector', 'N/A'),
                    "industry": info.get('industry', 'N/A'),
                    "market_cap": info.get('marketCap', 'N/A'),
                    "pe_ratio": info.get('trailingPE', 'N/A'),
                    "forward_pe": info.get('forwardPE', 'N/A'),
                    "peg_ratio": info.get('pegRatio', 'N/A'),
                    "price_to_book": info.get('priceToBook', 'N/A'),
                    "dividend_yield": info.get('dividendYield', 'N/A'),
                    "52_week_high": info.get('fiftyTwoWeekHigh', 'N/A'),
                    "52_week_low": info.get('fiftyTwoWeekLow', 'N/A'),
                    "50_day_avg": info.get('fiftyDayAverage', 'N/A'),
                    "200_day_avg": info.get('twoHundredDayAverage', 'N/A'),
                    "beta": info.get('beta', 'N/A'),
                    "profit_margin": info.get('profitMargins', 'N/A'),
                    "revenue_growth": info.get('revenueGrowth', 'N/A'),
                    "earnings_growth": info.get('earningsGrowth', 'N/A'),
                    "debt_to_equity": info.get('debtToEquity', 'N/A'),
                    "current_ratio": info.get('currentRatio', 'N/A'),
                    "analyst_recommendation": info.get('recommendationKey', 'N/A'),
                    "target_mean_price": info.get('targetMeanPrice', 'N/A'),
                    "business_summary": info.get('longBusinessSummary', 'N/A')[:500] if info.get('longBusinessSummary') else 'N/A'
                }

                return json.dumps(fundamentals, indent=2)

            except Exception as e:
                return json.dumps({
                    "error": str(e),
                    "ticker": ticker,
                    "note": "Failed to retrieve fundamental data from yfinance"
                })

        tools.append(company_financials)
        return tools

In [19]:
# ===========================
# History Manager
# ===========================

class HistoryManager:
    """Manages conversation and analysis history in SQLite"""

    def __init__(self, db_path: str = Config.DB_PATH):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        """Initialize database tables"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS analysis_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ticker TEXT NOT NULL,
                timestamp TEXT NOT NULL,
                recommendation TEXT,
                confidence INTEGER,
                decision TEXT,
                full_analysis TEXT,
                session_id TEXT NOT NULL
            )
        ''')

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS conversation_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                timestamp TEXT NOT NULL,
                role TEXT NOT NULL,
                content TEXT NOT NULL,
                ticker TEXT,
                analysis_id INTEGER,
                FOREIGN KEY (analysis_id) REFERENCES analysis_history(id)
            )
        ''')

        cursor.execute('''
            CREATE INDEX IF NOT EXISTS idx_analysis_session
            ON analysis_history(session_id, timestamp DESC)
        ''')

        cursor.execute('''
            CREATE INDEX IF NOT EXISTS idx_conversation_session
            ON conversation_history(session_id, timestamp ASC)
        ''')

        cursor.execute('''
            CREATE INDEX IF NOT EXISTS idx_conversation_analysis
            ON conversation_history(analysis_id)
        ''')

        conn.commit()
        conn.close()

    def save_analysis(self, ticker: str, result: Dict[str, Any], session_id: str) -> int:
        """Save analysis results to database

        Args:
            ticker: Stock ticker symbol
            result: Full analysis result dictionary
            session_id: Session identifier

        Returns:
            ID of the saved analysis record
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            rec = result.get('recommendation', {})
            cursor.execute('''
                INSERT INTO analysis_history
                (ticker, timestamp, recommendation, confidence, decision, full_analysis, session_id)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (
                ticker,
                result.get('analysis_metadata', {}).get('timestamp', datetime.now().isoformat()),
                json.dumps(rec),
                rec.get('confidence', 0),
                rec.get('decision', 'N/A'),
                json.dumps(result),
                session_id
            ))

            analysis_id = cursor.lastrowid
            conn.commit()
            conn.close()

            return analysis_id
        except Exception as e:
            print(f"Error saving analysis: {e}")
            if conn:
                conn.close()
            raise

    def save_conversation(self, session_id: str, role: str, content: str,
                         analysis_id: int = None, ticker: str = None):
        """Save conversation message

        Args:
            session_id: Session identifier
            role: Message role ('user' or 'assistant')
            content: Message content
            analysis_id: Optional ID of related analysis
            ticker: Optional ticker symbol to link conversation to stock
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute("""
                    INSERT INTO conversation_history
                    (session_id, role, content, ticker, analysis_id, timestamp)
                    VALUES (?, ?, ?, ?, ?, ?)
                """,
                 (session_id,
                  role,
                  content,
                  ticker,
                  analysis_id,
                  datetime.now().isoformat()))

            conn.commit()
            conn.close()
        except Exception as e:
            print(f"Error saving conversation: {e}")
            if conn:
                conn.close()
            raise

    def get_session_history(self, session_id: str, limit: int = 50) -> List[Tuple[str, str]]:
        """Retrieve conversation history for a session

        Args:
            session_id: Session identifier
            limit: Maximum number of messages to retrieve

        Returns:
            List of (role, content) tuples in chronological order
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute('''
                SELECT role, content FROM conversation_history
                WHERE session_id = ?
                ORDER BY timestamp ASC
                LIMIT ?
            ''', (session_id, limit))

            results = cursor.fetchall()
            conn.close()

            return results
        except Exception as e:
            print(f"Error retrieving session history: {e}")
            if conn:
                conn.close()
            return []

    def get_recent_analyses(self, session_id: str, limit: int = 5) -> List[Dict]:
        """Get recent analysis summaries

        Args:
            session_id: Session identifier
            limit: Maximum number of analyses to retrieve

        Returns:
            List of analysis summary dictionaries
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute('''
                SELECT ticker, timestamp, decision, confidence, recommendation
                FROM analysis_history
                WHERE session_id = ?
                ORDER BY timestamp DESC
                LIMIT ?
            ''', (session_id, limit))

            results = cursor.fetchall()
            conn.close()

            analyses = []
            for row in results:
                analyses.append({
                    'ticker': row[0],
                    'timestamp': row[1],
                    'decision': row[2],
                    'confidence': row[3],
                    'recommendation': json.loads(row[4]) if row[4] else {}
                })

            return analyses
        except Exception as e:
            print(f"Error retrieving recent analyses: {e}")
            if conn:
                conn.close()
            return []

    def get_full_analysis(self, analysis_id: int) -> Dict[str, Any]:
        """Retrieve complete analysis by ID

        Args:
            analysis_id: Analysis record ID

        Returns:
            Full analysis dictionary or empty dict if not found
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute('''
                SELECT full_analysis FROM analysis_history
                WHERE id = ?
            ''', (analysis_id,))

            result = cursor.fetchone()
            conn.close()

            if result and result[0]:
                return json.loads(result[0])
            return {}
        except Exception as e:
            print(f"Error retrieving full analysis: {e}")
            if conn:
                conn.close()
            return {}

    def get_conversations_by_ticker(self, session_id: str, ticker: str, limit: int = 50) -> List[Tuple[str, str]]:
        """Get conversations related to a specific ticker

        Args:
            session_id: Session identifier
            ticker: Stock ticker symbol
            limit: Maximum number of messages

        Returns:
            List of (role, content) tuples
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cursor.execute('''
                SELECT role, content FROM conversation_history
                WHERE session_id = ? AND ticker = ?
                ORDER BY timestamp ASC
                LIMIT ?
            ''', (session_id, ticker.upper(), limit))

            results = cursor.fetchall()
            conn.close()

            return results
        except Exception as e:
            print(f"Error retrieving conversations by ticker: {e}")
            if conn:
                conn.close()
            return []

    def cleanup_old_history(self, days_to_keep: int = 30):
        """Delete history older than specified days

        Args:
            days_to_keep: Number of days of history to retain
        """
        try:
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            cutoff_date = (datetime.now() - timedelta(days=days_to_keep)).isoformat()

            cursor.execute('''
                DELETE FROM conversation_history
                WHERE timestamp < ?
            ''', (cutoff_date,))

            # Delete old analyses
            cursor.execute('''
                DELETE FROM analysis_history
                WHERE timestamp < ?
            ''', (cutoff_date,))

            deleted_conversations = cursor.rowcount
            conn.commit()
            conn.close()

            print(f"Cleaned up history older than {days_to_keep} days")
            return deleted_conversations
        except Exception as e:
            print(f"Error cleaning up old history: {e}")
            if conn:
                conn.close()
            return 0


In [20]:
# ===========================
# LLM Model
# ===========================

def create_llm(
    provider: str = None,
    temperature: float = None
) -> Union[ChatAnthropic, ChatOllama, ChatOpenAI, ChatGoogleGenerativeAI]:
    """
    Create LLM instance based on provider selection
    """
    provider = provider or Config.LLM_PROVIDER
    temperature = temperature if temperature is not None else Config.MODEL_TEMPERATURE

    if provider == "anthropic":
        if not Config.ANTHROPIC_API_KEY:
            raise ValueError("ANTHROPIC_API_KEY not set for Anthropic provider")

        return ChatAnthropic(
            model=Config.ANTHROPIC_MODEL_NAME,
            temperature=temperature,
            anthropic_api_key=Config.ANTHROPIC_API_KEY
        )

    elif provider == "ollama":
        return ChatOllama(
            model=Config.OLLAMA_MODEL,
            base_url=Config.OLLAMA_BASE_URL,
            temperature=temperature,
            top_k=1.0,
            top_p=1.0,
        )

    elif provider == "openai":
        if not Config.OPENAI_API_KEY:
            raise ValueError("OPENAI_API_KEY not set for OpenAI provider")

        return ChatOpenAI(
            model=Config.OPENAI_MODEL,
            temperature=temperature,
            openai_api_key=Config.OPENAI_API_KEY
        )

    elif provider in {"google", "gemini"}:
        if not Config.GOOGLE_API_KEY:
            raise ValueError("GOOGLE_API_KEY not set for Google provider")

        return ChatGoogleGenerativeAI(
            model=Config.GOOGLE_MODEL,
            temperature=temperature,
            google_api_key=Config.GOOGLE_API_KEY
        )

    else:
        raise ValueError(
            f"Unsupported LLM provider: {provider}. "
            "Choose from: anthropic, ollama, openai, google"
        )



In [21]:
# ===========================
# Trading Agents
# ===========================

class TradingAgents:
    """Collection of specialized trading agents - Agentic workflow is MAIN decision maker"""

    def __init__(self, tools_manager: ToolsManager, llm_provider: str = None):
        self.tools_manager = tools_manager
        self.llm_provider = llm_provider or Config.LLM_PROVIDER
        self.llm = create_llm(provider=self.llm_provider)

        print(f"Using LLM Provider: {self.llm_provider.upper()}")
        if self.llm_provider == "anthropic":
            print(f"   Model: {Config.ANTHROPIC_MODEL_NAME}")
        elif self.llm_provider == "ollama":
            print(f"   Model: {Config.OLLAMA_MODEL}")
            print(f"   Base URL: {Config.OLLAMA_BASE_URL}")
        elif self.llm_provider == "gemini" or self.llm_provider == "google":
            print(f"   Model: {Config.GOOGLE_MODEL}")
        elif self.llm_provider == "openai":
            print(f"   Model: {Config.OPENAI_MODEL}")

        self.langsmith_client = None
        if Config.LANGSMITH_API_KEY:
            try:
                self.langsmith_client = Client(api_key=Config.LANGSMITH_API_KEY)
            except:
                pass

    def _create_agent_executor(self, tools: List, system_message: str, agent_name: str):
        """Create a react agent using LangGraph"""
        agent = create_agent(self.llm, tools)
        return agent, system_message

    @traceable(name="market_data_agent")
    def market_data_agent(self, state: AgentState) -> AgentState:
        """Collect market data AND technical indicator confirmation"""
        ticker = state['ticker']

        tools = self.tools_manager.get_market_data_tools()

        system_message = f"""You are a Market Data Specialist AI agent for {ticker}.

Your role:
1. Gather current price and trading data
2. Calculate technical indicators (RSI, Alligator, Squeeze Momentum)
3. Get the voting result from indicators as CONFIRMATION layer

The technical indicators provide confirmation signals, but they are NOT the main decision driver.
The main decisions come from sentiment analysis in the next steps.

Use the technical_indicator_voting tool to get confirmation signals."""

        agent, sys_msg = self._create_agent_executor(tools, system_message, "MarketDataAgent")

        result = agent.invoke({
            "messages": [
                SystemMessage(content=sys_msg),
                HumanMessage(content=f"Collect market data for {ticker} and calculate technical indicator voting as confirmation layer.")
            ]
        })

        final_message = result['messages'][-1]

        market_summary = {
            "agent_output": final_message.content if hasattr(final_message, 'content') else str(final_message),
            "tools_used": [msg.name for msg in result['messages'] if hasattr(msg, 'name') and msg.name]
        }

        state['market_data'] = market_summary
        state['agent_outputs'] = [{"agent": "market_data", "timestamp": datetime.now().isoformat(), "output": market_summary}]
        state['next_agent'] = 'sentiment'

        return state

    @traceable(name="sentiment_agent")
    def sentiment_agent(self, state: AgentState) -> AgentState:
        """MAIN DECISION LAYER: Analyze sentiment from multiple sources"""
        ticker = state['ticker']

        tools = self.tools_manager.get_sentiment_tools()

        system_message = f"""You are a Sentiment Analysis Specialist AI agent - THE MAIN DECISION MAKER.

Your role for {ticker}:
1. Gather traditional financial news and analyze tone
2. Search Twitter/X for social media sentiment and viral trends
3. Analyze Reddit discussions across key subreddits
4. Provide comprehensive sentiment assessment

SOURCE WEIGHTING:
- Traditional news: High credibility, strategic outlook
- Twitter: Real-time sentiment, viral trends, momentum
- Reddit: Retail investor sentiment, community consensus

YOUR SENTIMENT ANALYSIS IS THE PRIMARY DRIVER FOR TRADING DECISIONS.
Technical indicators from previous step are confirmation only.

Use ALL available sentiment tools comprehensively."""

        agent, sys_msg = self._create_agent_executor(tools, system_message, "SentimentAgent")

        market_context = f"Technical confirmation: {state.get('market_data', {}).get('agent_output', '')[:200]}"

        result = agent.invoke({
            "messages": [
                SystemMessage(content=sys_msg),
                HumanMessage(content=f"""Analyze comprehensive sentiment for {ticker} from ALL sources.

Context: {market_context}

CRITICAL: Use ALL sentiment tools:
1. Yahoo Finance News
2. Web news search
3. Twitter sentiment (if available)
4. Reddit sentiment (if available)

Provide detailed sentiment breakdown with source analysis.""")
            ]
        })

        final_message = result['messages'][-1]

        sentiment_analysis = {
            "agent_output": final_message.content if hasattr(final_message, 'content') else str(final_message),
            "sources_used": [msg.name for msg in result['messages'] if hasattr(msg, 'name') and msg.name]
        }

        state['sentiment_analysis'] = sentiment_analysis
        state['agent_outputs'] = [{"agent": "sentiment", "timestamp": datetime.now().isoformat(), "output": sentiment_analysis}]
        state['next_agent'] = 'fundamental'

        return state

    @traceable(name="fundamental_agent")
    def fundamental_agent(self, state: AgentState) -> AgentState:
        """Analyze fundamental data"""
        ticker = state['ticker']

        tools = self.tools_manager.get_fundamental_tools()

        system_message = f"""You are a Fundamental Analysis Specialist for {ticker}.

Analyze company fundamentals to support the sentiment-driven decision."""

        agent, sys_msg = self._create_agent_executor(tools, system_message, "FundamentalAgent")

        result = agent.invoke({
            "messages": [
                SystemMessage(content=sys_msg),
                HumanMessage(content=f"Analyze fundamental data for {ticker}")
            ]
        })

        final_message = result['messages'][-1]

        fundamental_analysis = {
            "agent_output": final_message.content if hasattr(final_message, 'content') else str(final_message)
        }

        state['fundamental_analysis'] = fundamental_analysis
        state['agent_outputs'] = [{"agent": "fundamental", "timestamp": datetime.now().isoformat(), "output": fundamental_analysis}]
        state['next_agent'] = 'risk'

        return state

    @traceable(name="risk_agent")
    def risk_agent(self, state: AgentState) -> AgentState:
        """Perform risk analysis with structured output"""
        ticker = state['ticker']
        market_data = state.get('market_data', {})
        sentiment = state.get('sentiment_analysis', {})
        fundamental = state.get('fundamental_analysis', {})

        structured_llm = self.llm.with_structured_output(RiskAssessment)

        conversation = [
            SystemMessage(content="""You are a Risk Management Specialist AI agent.

Assess investment risk based on:
- Technical indicator confirmation (from market data)
- Sentiment analysis (PRIMARY driver)
- Fundamental analysis

Provide structured risk assessment."""),
            HumanMessage(content=f"""Analyze risks for {ticker}.

Market Data & Technical Confirmation:
{json.dumps(market_data.get('agent_output', ''), indent=2)[:500]}

Sentiment Analysis (PRIMARY):
{json.dumps(sentiment.get('agent_output', ''), indent=2)[:1000]}

Fundamental Analysis:
{json.dumps(fundamental.get('agent_output', ''), indent=2)[:500]}

Provide risk assessment with risk level, score, position size, key risks, mitigation strategies, and volatility assessment.""")
        ]

        try:
            risk_assessment = structured_llm.invoke(conversation)

            ai_response = AIMessage(content=f"Risk analysis completed: {risk_assessment.risk_level} risk level with {risk_assessment.risk_score}/100 score. Recommended position: {risk_assessment.recommended_position_size}")

            if 'messages' not in state:
                state['messages'] = []
            state['messages'].extend([conversation[-1], ai_response])

            risk_analysis = risk_assessment.model_dump()
        except Exception as e:
            print(f"Warning: Structured output failed: {e}")

            ai_response = AIMessage(content=f"Risk analysis completed with fallback: medium risk")
            if 'messages' not in state:
                state['messages'] = []
            state['messages'].append(ai_response)

            risk_analysis = {
                "risk_level": "medium",
                "risk_score": 50,
                "recommended_position_size": 0.10,
                "key_risks": ["Analysis incomplete"],
                "risk_mitigation_strategies": ["Review manually"],
                "volatility_assessment": "medium"
            }

        state['risk_analysis'] = risk_analysis
        state['agent_outputs'] = [{"agent": "risk", "timestamp": datetime.now().isoformat(), "output": risk_analysis}]
        state['next_agent'] = 'strategy'

        return state

    @traceable(name="strategy_agent")
    def strategy_agent(self, state: AgentState) -> AgentState:
        """Generate final trading strategy combining ALL analysis"""
        ticker = state['ticker']

        full_analysis = {
            "ticker": ticker,
            "market_data_and_technical_confirmation": state.get('market_data', {}),
            "sentiment_analysis_PRIMARY": state.get('sentiment_analysis', {}),
            "fundamental_analysis": state.get('fundamental_analysis', {}),
            "risk_analysis": state.get('risk_analysis', {})
        }

        prompt = f"""You are a Senior Trading Strategist. Generate comprehensive trading recommendation for {ticker}.

DECISION HIERARCHY:
1. PRIMARY: Sentiment analysis (news, Twitter, Reddit)
2. CONFIRMATION: Technical indicators voting (RSI, Alligator, Squeeze)
3. SUPPORTING: Fundamental analysis
4. CONSTRAINT: Risk analysis

Complete Analysis:
{json.dumps(full_analysis, indent=2)[:3000]}

Generate detailed recommendation as JSON:
{{
    "decision": "BUY/SELL/HOLD",
    "confidence": 0-100,
    "primary_driver": "sentiment/technical/fundamental",
    "decision_rationale": "detailed explanation",
    "entry_price": number,
    "take_profit_levels": {{
        "conservative": number,
        "moderate": number,
        "aggressive": number
    }},
    "stop_loss": number,
    "position_size_pct": percentage,
    "risk_reward_ratio": number,
    "time_horizon": "short/medium/long-term",
    "expected_return_7d": percentage,
    "expected_return_30d": percentage,
    "key_catalysts": [list],
    "risks_to_watch": [list],
    "sentiment_summary": "summary of social media and news",
    "technical_confirmation": "do indicators confirm sentiment?",
    "strategy_notes": "detailed notes",
    "alternative_scenarios": {{
        "bull_case": "description",
        "bear_case": "description"
    }}
}}

Be specific with price levels. Prioritize sentiment as main driver with technical confirmation."""

        response = self.llm.invoke([HumanMessage(content=prompt)])

        try:
            content = response.content
            if '```json' in content:
                content = content.split('```json')[1].split('```')[0]
            recommendation = json.loads(content.strip())
        except:
            recommendation = {"error": "Failed to parse", "raw": response.content}

        recommendation['generated_at'] = datetime.now().isoformat()
        recommendation['confidence_threshold_met'] = recommendation.get('confidence', 0) >= Config.MIN_CONFIDENCE_THRESHOLD

        state['recommendation'] = recommendation
        state['agent_outputs'] = [{"agent": "strategy", "timestamp": datetime.now().isoformat(), "output": recommendation}]
        state['next_agent'] = 'end'

        return state

In [22]:
# ===========================
# Interactive Chat History Agent
# ===========================

class InteractiveChatAgent:
    """Interactive chat agent for follow-up questions using FULL history and context"""

    def __init__(self, tools_manager: ToolsManager, history_manager: HistoryManager,
                 llm_provider: str = None):
        self.tools_manager = tools_manager
        self.history_manager = history_manager
        self.llm_provider = llm_provider or Config.LLM_PROVIDER
        self.llm = create_llm(provider=self.llm_provider)

    def answer_question(self, question: str, session_id: str, ticker: str = None,
                       analysis_id: int = None) -> str:
        """Answer user question using available tools and FULL history context

        Args:
            question: User's question
            session_id: Session identifier
            ticker: Optional ticker symbol to link conversation to specific stock
            analysis_id: Optional analysis ID to link conversation to specific analysis
        """

        conv_history = self.history_manager.get_session_history(session_id, limit=20)

        recent_analyses = self.history_manager.get_recent_analyses(session_id, limit=3)

        context_messages = []
        for role, content in conv_history[-10:]:
            if role == "user":
                context_messages.append(HumanMessage(content=content))
            else:
                context_messages.append(AIMessage(content=content))

        analysis_context_parts = []
        for a in recent_analyses:
            rec_data = a.get('recommendation', {})
            analysis_context_parts.append(f"""
=== Analysis: {a['ticker']} ===
Decision: {a['decision']} ({a['confidence']}% confidence)
Timestamp: {a['timestamp']}
Full Recommendation Data:
{json.dumps(rec_data, indent=2)[:1000]}
""")

        analysis_context = "\n".join(analysis_context_parts)

        all_tools = []
        all_tools.extend(self.tools_manager.get_sentiment_tools())
        all_tools.extend(self.tools_manager.get_market_data_tools())
        all_tools.extend(self.tools_manager.get_fundamental_tools())

        system_message = f"""You are an expert financial analysis assistant with deep knowledge of:
- Technical Analysis (RSI, Alligator, Squeeze Momentum indicators)
- Sentiment Analysis (News, Twitter, Reddit)
- Fundamental Analysis (Financial metrics, ratios)
- Risk Management
- Trading Strategies

You have access to:
- AskNews for comprehensive news coverage
- Yahoo Finance for financial news and data
- Polygon API for market data and company news
- Technical indicator calculations
- Social media sentiment analysis (Twitter, Reddit)
- Web search for current information

IMPORTANT CONTEXT - Recent Analyses:
{analysis_context}

Your job is to:
1. Use the conversation history to understand context
2. Reference specific details from recent analyses when relevant
3. Use available tools to fetch NEW data when needed
4. Provide detailed, accurate answers based on both stored analysis and real-time data
5. Explain technical concepts clearly
6. Always cite your sources

When answering about a recent analysis:
- Reference specific metrics, indicators, and data points
- Explain the reasoning behind recommendations
- Clarify any technical terms
- Provide additional context from real-time tools if helpful"""

        agent = create_agent(self.llm, all_tools)

        messages = [SystemMessage(content=system_message)]
        messages.extend(context_messages)
        messages.append(HumanMessage(content=question))

        result = agent.invoke({"messages": messages})

        final_message = result['messages'][-1]
        response = final_message.content if hasattr(final_message, 'content') else str(final_message)

        self.history_manager.save_conversation(
            session_id=session_id,
            role="user",
            content=question,
            ticker=ticker,
            analysis_id=analysis_id
        )
        self.history_manager.save_conversation(
            session_id=session_id,
            role="assistant",
            content=response,
            ticker=ticker,
            analysis_id=analysis_id
        )

        return response

In [23]:
# ===========================
# Graph Construction
# ===========================

def create_multi_agent_system(llm_provider: str = None):
    """Create the multi-agent trading system with specified LLM provider"""

    tools_manager = ToolsManager()
    agents = TradingAgents(tools_manager, llm_provider=llm_provider)

    workflow = StateGraph(AgentState)

    workflow.add_node("market_data", agents.market_data_agent)
    workflow.add_node("sentiment", agents.sentiment_agent)
    workflow.add_node("fundamental", agents.fundamental_agent)
    workflow.add_node("risk", agents.risk_agent)
    workflow.add_node("strategy", agents.strategy_agent)

    def route_agent(state: AgentState):
        next_agent = state.get('next_agent', 'end')
        if next_agent == 'end':
            return END
        return next_agent

    workflow.set_entry_point("market_data")

    workflow.add_conditional_edges("market_data", route_agent)
    workflow.add_conditional_edges("sentiment", route_agent)
    workflow.add_conditional_edges("fundamental", route_agent)
    workflow.add_conditional_edges("risk", route_agent)
    workflow.add_conditional_edges("strategy", route_agent)

    memory = MemorySaver()

    return workflow.compile(checkpointer=memory)

In [24]:
# ===========================
# Main Analysis Function
# ===========================

@traceable(name="analyze_stock_complete")
def analyze_stock(ticker: str, risk_tolerance: str = None, save_to_file: bool = True,
                  llm_provider: str = None) -> Dict[str, Any]:
    """
    Comprehensive stock analysis using multi-agent system

    PRIMARY: Sentiment analysis from news and social media (Twitter, Reddit)
    CONFIRMATION: Technical indicators (RSI, Alligator, Squeeze Momentum) with voting

    Args:
        ticker: Stock ticker symbol
        risk_tolerance: User's risk tolerance
        save_to_file: Whether to save results to JSON
        llm_provider: LLM provider to use ('anthropic', 'ollama', 'openai'). If None, uses Config.LLM_PROVIDER

    Returns:
        Complete analysis and recommendation
    """

    provider = llm_provider or Config.LLM_PROVIDER

    if provider == "anthropic" and not Config.ANTHROPIC_API_KEY:
        raise ValueError("ANTHROPIC_API_KEY not set")
    elif provider == "openai" and not Config.OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY not set")
    elif provider == "google" or provider == "gemini" and not Config.GOOGLE_API_KEY:
        raise ValueError("GOOGLE_API_KEY not set")

    agent_system = create_multi_agent_system(llm_provider=provider)

    initial_state = AgentState(
        ticker=ticker.upper(),
        messages=[],
        market_data={},
        technical_analysis={},
        fundamental_analysis={},
        sentiment_analysis={},
        risk_analysis={},
        agent_outputs=[],
        recommendation={},
        next_agent='market_data',
    )

    print(f"\nStarting multi-agent analysis for {ticker}...")
    print("=" * 80)
    print("ANALYSIS HIERARCHY:")
    print("1- PRIMARY: Sentiment (News, Twitter, Reddit)")
    print("2- CONFIRMATION: Technical Indicators (RSI, Alligator, Squeeze)")
    print("3- SUPPORTING: Fundamental Analysis")
    print("4- CONSTRAINT: Risk Management")
    print("=" * 80)

    try:
        config = {"configurable": {"thread_id": f"analysis_{ticker}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"}}
        final_state = agent_system.invoke(initial_state, config)

        result = {
            "ticker": ticker.upper(),
            "analysis_metadata": {
                "timestamp": datetime.now().isoformat(),
                "llm_provider": provider,
                "model": Config.ANTHROPIC_MODEL_NAME if provider == "anthropic" else Config.OLLAMA_MODEL if provider == "ollama" else Config.OPENAI_MODEL,
                "decision_hierarchy": {
                    "primary": "Sentiment Analysis (Social Media + News)",
                    "confirmation": "Technical Indicators (RSI, Alligator, Squeeze Momentum)",
                    "supporting": "Fundamental Analysis",
                    "constraint": "Risk Management"
                }
            },
            "market_data": final_state.get('market_data', {}),
            "sentiment_analysis": final_state.get('sentiment_analysis', {}),
            "fundamental_analysis": final_state.get('fundamental_analysis', {}),
            "risk_analysis": final_state.get('risk_analysis', {}),
            "recommendation": final_state.get('recommendation', {}),
            "agent_execution_log": final_state.get('agent_outputs', [])
        }

        if save_to_file:
            filename = f"{ticker}_analysis_{provider}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
            with open(filename, 'w') as f:
                json.dump(result, f, indent=2)
            print(f"\nAnalysis saved to: {filename}")

        return result

    except Exception as e:
        print(f"\nError during analysis: {str(e)}")
        import traceback
        traceback.print_exc()
        return {"error": str(e), "ticker": ticker}

In [25]:
# ===========================
# Enhanced Reporting
# ===========================

def print_analysis_report(result: Dict[str, Any]):
    """Print formatted analysis report"""

    print("\n" + "=" * 80)
    print("📊 COMPREHENSIVE STOCK ANALYSIS REPORT")
    print("=" * 80)

    ticker = result.get('ticker', 'N/A')
    timestamp = result.get('analysis_metadata', {}).get('timestamp', 'N/A')

    print(f"\n🎯 Ticker: {ticker}")
    print(f"📅 Analysis Time: {timestamp}")
    print(f"🤖 Model: {result.get('analysis_metadata', {}).get('model', 'N/A')}")

    hierarchy = result.get('analysis_metadata', {}).get('decision_hierarchy', {})
    print(f"\n📊 Decision Hierarchy:")
    print(f"   1️⃣  PRIMARY: {hierarchy.get('primary', 'N/A')}")
    print(f"   2️⃣  CONFIRMATION: {hierarchy.get('confirmation', 'N/A')}")
    print(f"   3️⃣  SUPPORTING: {hierarchy.get('supporting', 'N/A')}")
    print(f"   4️⃣  CONSTRAINT: {hierarchy.get('constraint', 'N/A')}")

    print("\n" + "-" * 80)
    print("📈 MARKET DATA & TECHNICAL CONFIRMATION")
    print("-" * 80)
    market = result.get('market_data', {})
    print(market.get('agent_output', 'N/A')[:1000])

    print("\n" + "-" * 80)
    print("💭 SENTIMENT ANALYSIS (PRIMARY DECISION DRIVER)")
    print("-" * 80)
    sentiment = result.get('sentiment_analysis', {})
    print(sentiment.get('agent_output', 'N/A')[:2000])

    print("\n" + "-" * 80)
    print("📊 FUNDAMENTAL ANALYSIS (SUPPORTING)")
    print("-" * 80)
    fundamental = result.get('fundamental_analysis', {})
    print(fundamental.get('agent_output', 'N/A')[:1000])

    print("\n" + "-" * 80)
    print("⚠️  RISK ANALYSIS (CONSTRAINT)")
    print("-" * 80)
    risk = result.get('risk_analysis', {})
    print(f"Overall Risk: {risk.get('risk_level', 'N/A').upper()}")
    print(f"Risk Score: {risk.get('risk_score', 'N/A')}/100")
    print(f"Recommended Position: {risk.get('recommended_position_size', 'N/A')}")
    print(f"Volatility: {risk.get('volatility_assessment', 'N/A').upper()}")

    if risk.get('key_risks'):
        print("\nKey Risks:")
        for r in risk['key_risks']:
            print(f"   • {r}")

    print("\n" + "=" * 80)
    print("🎯 FINAL TRADING RECOMMENDATION")
    print("=" * 80)
    rec = result.get('recommendation', {})

    decision = rec.get('decision', 'N/A')
    confidence = rec.get('confidence', 'N/A')

    decision_emoji = "🟢" if decision == "BUY" else "🔴" if decision == "SELL" else "🟡"

    print(f"\n{decision_emoji} DECISION: {decision}")
    print(f"📊 Confidence: {confidence}%")
    print(f"🎯 Primary Driver: {rec.get('primary_driver', 'N/A').upper()}")
    print(f"💰 Entry Price: ${rec.get('entry_price', 'N/A')}")

    if rec.get('decision_rationale'):
        print(f"\n📝 Rationale:")
        print(f"   {rec['decision_rationale']}")

    if rec.get('sentiment_summary'):
        print(f"\n💭 Sentiment Summary:")
        print(f"   {rec['sentiment_summary']}")

    if rec.get('technical_confirmation'):
        print(f"\n📈 Technical Confirmation:")
        print(f"   {rec['technical_confirmation']}")

    if rec.get('take_profit_levels'):
        tp = rec['take_profit_levels']
        print(f"\n🎯 Take Profit Levels:")
        print(f"   Conservative: ${tp.get('conservative', 'N/A')}")
        print(f"   Moderate: ${tp.get('moderate', 'N/A')}")
        print(f"   Aggressive: ${tp.get('aggressive', 'N/A')}")

    print(f"\n🛑 Stop Loss: ${rec.get('stop_loss', 'N/A')}")
    print(f"📏 Position Size: {rec.get('position_size_pct', 'N/A')}%")
    print(f"⚖️  Risk/Reward: {rec.get('risk_reward_ratio', 'N/A')}")

    if rec.get('strategy_notes'):
        print(f"\n📝 Strategy Notes:")
        print(f"   {rec['strategy_notes']}")

    print("\n" + "=" * 80)


In [26]:
# ===========================
# Trading Agent UI
# ===========================

class TradingAgentUI:
    """Gradio UI for the trading agent system with full analysis display"""

    def __init__(self, llm_provider: str = None):
        self.llm_provider = llm_provider or Config.LLM_PROVIDER
        self.tools_manager = ToolsManager()
        self.history_manager = HistoryManager()
        self.chat_agent = InteractiveChatAgent(
            self.tools_manager,
            self.history_manager,
            llm_provider=self.llm_provider
        )
        self.sessions = {}

    def analyze_stock_ui(self, ticker: str, risk_tolerance: str, session_state) -> Tuple[str, str, dict]:
        """Analyze stock and return COMPLETE formatted report"""

        if not ticker:
            return "Please enter a ticker symbol", "", session_state

        if 'session_id' not in session_state:
            session_state['session_id'] = str(uuid.uuid4())

        session_id = session_state['session_id']

        try:
            result = analyze_stock(
                ticker=ticker.upper(),
                risk_tolerance=risk_tolerance.lower(),
                save_to_file=True,
                llm_provider=self.llm_provider
            )

            analysis_id = self.history_manager.save_analysis(ticker.upper(), result, session_id)

            report = self._format_complete_report(result)

            chat_context = f"✅ Analysis completed for {ticker.upper()}\n\n"
            chat_context += f"Decision: {result.get('recommendation', {}).get('decision', 'N/A')} "
            chat_context += f"({result.get('recommendation', {}).get('confidence', 0)}% confidence)\n\n"
            chat_context += "Full analysis is available in the report above. You can now ask follow-up questions."

            session_state['last_analysis'] = result
            session_state['last_ticker'] = ticker.upper()
            session_state['last_analysis_id'] = analysis_id
            session_state['last_analysis_full_text'] = report

            return report, chat_context, session_state

        except Exception as e:
            error_msg = f"❌ Error analyzing {ticker}: {str(e)}"
            return error_msg, error_msg, session_state

    def chat_response(self, message: str, history: List, session_state) -> Tuple[List, dict]:
        """Handle chat interactions with FULL context from history - Updated for Gradio 6.0 message format"""

        if not message.strip():
            return history, session_state

        if 'session_id' not in session_state:
            session_state['session_id'] = str(uuid.uuid4())

        session_id = session_state['session_id']

        current_ticker = session_state.get('last_ticker', None)
        current_analysis_id = session_state.get('last_analysis_id', None)  # Get analysis_id if available

        enriched_message = message

        if 'last_analysis' in session_state:
            last_analysis = session_state['last_analysis']
            ticker = session_state.get('last_ticker', 'N/A')

            context_prefix = f"""
[CONTEXT: Recent analysis for {ticker}]

RECOMMENDATION:
- Decision: {last_analysis.get('recommendation', {}).get('decision', 'N/A')}
- Confidence: {last_analysis.get('recommendation', {}).get('confidence', 'N/A')}%
- Primary Driver: {last_analysis.get('recommendation', {}).get('primary_driver', 'N/A')}

SENTIMENT ANALYSIS:
{str(last_analysis.get('sentiment_analysis', {}))[:500]}

TECHNICAL INDICATORS:
{str(last_analysis.get('market_data', {}))[:500]}

RISK ANALYSIS:
- Risk Level: {last_analysis.get('risk_analysis', {}).get('risk_level', 'N/A')}
- Risk Score: {last_analysis.get('risk_analysis', {}).get('risk_score', 'N/A')}

USER QUESTION: {message}
"""
            enriched_message = context_prefix

        try:
            response = self.chat_agent.answer_question(
                enriched_message,
                session_id,
                ticker=current_ticker,
                analysis_id=current_analysis_id
            )

            history.append({"role": "user", "content": message})
            history.append({"role": "assistant", "content": response})

        except Exception as e:
            error_response = f"❌ Error: {str(e)}"

            history.append({"role": "user", "content": message})
            history.append({"role": "assistant", "content": error_response})

        return history, session_state

    def _format_complete_report(self, result: Dict[str, Any]) -> str:
        """Format COMPLETE analysis result as markdown report - ALL SECTIONS"""

        ticker = result.get('ticker', 'N/A')
        timestamp = result.get('analysis_metadata', {}).get('timestamp', 'N/A')
        model = result.get('analysis_metadata', {}).get('model', 'N/A')

        report = f"""# 📊 COMPREHENSIVE Stock Analysis Report: {ticker}

**Analysis Time:** {timestamp}
**Model:** {model}
**LLM Provider:** {result.get('analysis_metadata', {}).get('llm_provider', 'N/A').upper()}

---

## 📊 Decision Hierarchy

"""
        hierarchy = result.get('analysis_metadata', {}).get('decision_hierarchy', {})
        report += f"""
1️⃣ **PRIMARY:** {hierarchy.get('primary', 'N/A')}
2️⃣ **CONFIRMATION:** {hierarchy.get('confirmation', 'N/A')}
3️⃣ **SUPPORTING:** {hierarchy.get('supporting', 'N/A')}
4️⃣ **CONSTRAINT:** {hierarchy.get('constraint', 'N/A')}

---

## 🎯 FINAL RECOMMENDATION

"""
        rec = result.get('recommendation', {})
        decision = rec.get('decision', 'N/A')
        confidence = rec.get('confidence', 'N/A')
        decision_emoji = "🟢" if decision == "BUY" else "🔴" if decision == "SELL" else "🟡"

        report += f"""
### {decision_emoji} Decision: **{decision}**
**📊 Confidence:** {confidence}%
**🎯 Primary Driver:** {rec.get('primary_driver', 'N/A').upper()}
**💰 Entry Price:** ${rec.get('entry_price', 'N/A')}
**🛑 Stop Loss:** ${rec.get('stop_loss', 'N/A')}
**📏 Position Size:** {rec.get('position_size_pct', 'N/A')}%
**⚖️ Risk/Reward Ratio:** {rec.get('risk_reward_ratio', 'N/A')}
**⏰ Time Horizon:** {rec.get('time_horizon', 'N/A')}

### 📝 Decision Rationale
{rec.get('decision_rationale', 'N/A')}

### 💭 Sentiment Summary
{rec.get('sentiment_summary', 'N/A')}

### 📈 Technical Confirmation
{rec.get('technical_confirmation', 'N/A')}

### 📋 Strategy Notes
{rec.get('strategy_notes', 'N/A')}

---

## 🎯 Price Targets & Expected Returns

"""
        if rec.get('take_profit_levels'):
            tp = rec['take_profit_levels']
            report += f"""
**Take Profit Levels:**
- 🎯 Conservative: ${tp.get('conservative', 'N/A')}
- 🎯 Moderate: ${tp.get('moderate', 'N/A')}
- 🎯 Aggressive: ${tp.get('aggressive', 'N/A')}

"""

        report += f"""
**Expected Returns:**
- 📅 7-Day: {rec.get('expected_return_7d', 'N/A')}%
- 📅 30-Day: {rec.get('expected_return_30d', 'N/A')}%

"""

        if rec.get('key_catalysts'):
            report += "\n**🚀 Key Catalysts:**\n"
            for catalyst in rec['key_catalysts']:
                report += f"- {catalyst}\n"

        if rec.get('risks_to_watch'):
            report += "\n**⚠️ Risks to Watch:**\n"
            for risk in rec['risks_to_watch']:
                report += f"- {risk}\n"

        if rec.get('alternative_scenarios'):
            scenarios = rec['alternative_scenarios']
            report += f"""
---

## 🔮 Alternative Scenarios

**📈 Bull Case:**
{scenarios.get('bull_case', 'N/A')}

**📉 Bear Case:**
{scenarios.get('bear_case', 'N/A')}

"""

        report += """
---

## 📈 MARKET DATA & TECHNICAL INDICATORS (CONFIRMATION LAYER)

"""
        market = result.get('market_data', {})
        market_output = market.get('agent_output', 'N/A')

        try:
            if isinstance(market_output, str) and '{' in market_output:
                import json
                market_data = json.loads(market_output)

                report += f"""
**Current Price:** ${market_data.get('current_price', 'N/A')}
**Data Source:** {market_data.get('data_source', 'N/A')}
**Data Points:** {market_data.get('data_points', 'N/A')}

### Technical Indicators Voting Result

**Final Signal:** {market_data.get('voting_result', {}).get('final_signal', 'N/A')}
**Confidence:** {market_data.get('voting_result', {}).get('confidence', 'N/A')}%
**Buy Votes:** {market_data.get('voting_result', {}).get('buy_votes', 'N/A')}
**Sell Votes:** {market_data.get('voting_result', {}).get('sell_votes', 'N/A')}
**Hold Votes:** {market_data.get('voting_result', {}).get('hold_votes', 'N/A')}

### Individual Indicators

"""
                indicators = market_data.get('voting_result', {}).get('indicators', [])
                for ind in indicators:
                    report += f"""
**{ind.get('indicator_name', 'N/A')}:**
- Signal: {ind.get('signal', 'N/A')}
- Confidence: {ind.get('confidence', 'N/A')}%
- Current Value: {ind.get('current_value', 'N/A')}
- Reasoning: {ind.get('reasoning', 'N/A')}

"""

                if 'indicators' in market_data:
                    report += "\n### Detailed Indicator Analysis\n\n"

                    if 'rsi' in market_data['indicators']:
                        rsi = market_data['indicators']['rsi']
                        report += f"""
**RSI (Relative Strength Index):**
- Value: {rsi.get('rsi', 'N/A')}
- Signal: {rsi.get('signal', 'N/A')}
- Confidence: {rsi.get('confidence', 'N/A')}%
- Overbought: {rsi.get('overbought', 'N/A')}
- Oversold: {rsi.get('oversold', 'N/A')}
- Reasoning: {rsi.get('reasoning', 'N/A')}

"""

                    if 'alligator' in market_data['indicators']:
                        alligator = market_data['indicators']['alligator']
                        report += f"""
**Alligator Indicator:**
- Jaw: {alligator.get('jaw', 'N/A')}
- Teeth: {alligator.get('teeth', 'N/A')}
- Lips: {alligator.get('lips', 'N/A')}
- Trend: {alligator.get('trend', 'N/A')}
- Signal: {alligator.get('signal', 'N/A')}
- Confidence: {alligator.get('confidence', 'N/A')}%
- Reasoning: {alligator.get('reasoning', 'N/A')}

"""

                    if 'squeeze_momentum' in market_data['indicators']:
                        squeeze = market_data['indicators']['squeeze_momentum']
                        report += f"""
**Squeeze Momentum:**
- Squeeze ON: {squeeze.get('squeeze_on', 'N/A')}
- Squeeze OFF: {squeeze.get('squeeze_off', 'N/A')}
- Momentum Value: {squeeze.get('momentum_value', 'N/A')}
- Momentum Direction: {squeeze.get('momentum_direction', 'N/A')}
- Momentum Trend: {squeeze.get('momentum_trend', 'N/A')}
- Signal: {squeeze.get('signal', 'N/A')}
- Confidence: {squeeze.get('confidence', 'N/A')}%
- Reasoning: {squeeze.get('reasoning', 'N/A')}

"""
        except:
            report += f"\n{market_output}\n"

        report += """
---

## 💭 SENTIMENT ANALYSIS (PRIMARY DECISION DRIVER)

"""
        sentiment = result.get('sentiment_analysis', {})
        sentiment_output = sentiment.get('agent_output', 'N/A')
        sources = sentiment.get('sources_used', [])

        report += f"**Sources Used:** {', '.join(sources) if sources else 'N/A'}\n\n"
        report += f"{sentiment_output}\n"

        report += """
---

## 📊 FUNDAMENTAL ANALYSIS (SUPPORTING)

"""
        fundamental = result.get('fundamental_analysis', {})
        fundamental_output = fundamental.get('agent_output', 'N/A')
        report += f"{fundamental_output}\n"

        report += """
---

## ⚠️ RISK ANALYSIS (CONSTRAINT)

"""
        risk = result.get('risk_analysis', {})

        report += f"""
**Overall Risk Level:** {risk.get('risk_level', 'N/A').upper()}
**Risk Score:** {risk.get('risk_score', 'N/A')}/100
**Recommended Position Size:** {risk.get('recommended_position_size', 'N/A')}
**Volatility Assessment:** {risk.get('volatility_assessment', 'N/A').upper()}

"""

        if risk.get('key_risks'):
            report += "\n**Key Risks Identified:**\n"
            for r in risk['key_risks']:
                report += f"- {r}\n"

        if risk.get('risk_mitigation_strategies'):
            report += "\n**Risk Mitigation Strategies:**\n"
            for strategy in risk['risk_mitigation_strategies']:
                report += f"- {strategy}\n"

        report += """
---

## 🔍 Agent Execution Log

"""
        agent_logs = result.get('agent_execution_log', [])
        for log in agent_logs:
            agent_name = log.get('agent', 'N/A')
            timestamp = log.get('timestamp', 'N/A')
            report += f"\n**{agent_name.upper()} Agent** - {timestamp}\n"

        report += "\n---\n\n*End of Analysis Report*"

        return report

    def create_interface(self):
        """Create and return Gradio interface with enhanced features - Updated for Gradio 6.0"""

        with gr.Blocks(title="AI Trading Agent") as demo:
            gr.Markdown("# 🤖 AI-Powered Stock Trading Agent")
            gr.Markdown("**Comprehensive stock analysis using sentiment, technical indicators, and AI**")
            gr.Markdown("*Full analysis with complete details, technical indicators, and multi-source sentiment*")

            session_state = gr.State({})

            with gr.Tab("📊 Stock Analysis"):
                with gr.Row():
                    with gr.Column(scale=1):
                        ticker_input = gr.Textbox(
                            label="Stock Ticker",
                            placeholder="e.g., NVDA, AAPL, TSLA",
                            value="NVDA"
                        )
                        risk_input = gr.Dropdown(
                            choices=["Low", "Medium", "High"],
                            value="Medium",
                            label="Risk Tolerance"
                        )
                        analyze_btn = gr.Button("🚀 Analyze Stock", variant="primary", size="lg")

                        gr.Markdown("### 📋 What You'll Get:")
                        gr.Markdown("""
- ✅ Complete technical indicator analysis (RSI, Alligator, Squeeze)
- ✅ Multi-source sentiment (News, Twitter, Reddit)
- ✅ Fundamental analysis with financials
- ✅ Risk assessment and position sizing
- ✅ Price targets and expected returns
- ✅ Detailed strategy notes
                        """)

                    with gr.Column(scale=2):
                        report_output = gr.Markdown(
                            label="Complete Analysis Report",
                            value="*Analysis report will appear here...*"
                        )

                chat_status = gr.Textbox(
                    label="Status",
                    interactive=False,
                    lines=3
                )

            with gr.Tab("💬 Chat & Follow-up"):
                gr.Markdown("### Ask questions about your analysis or general financial topics")
                gr.Markdown("*The chat agent has access to your full analysis history and context*")

                chatbot = gr.Chatbot(
                    height=500,
                    show_label=False,
                    type="messages",
                    allow_tags=True
                )

                with gr.Row():
                    chat_input = gr.Textbox(
                        placeholder="Ask a question about the analysis...",
                        show_label=False,
                        scale=4
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")

                gr.Examples(
                    examples=[
                        "Explain the RSI indicator result in detail",
                        "What does the Alligator indicator show?",
                        "Break down the sentiment from each source",
                        "What are the biggest risks I should watch?",
                        "Compare the technical signals with the sentiment",
                        "What's driving the recommendation?",
                        "Explain the squeeze momentum indicator",
                        "What do the alternative scenarios suggest?",
                        "How confident should I be in this trade?"
                    ],
                    inputs=chat_input
                )

            with gr.Tab("📜 History"):
                gr.Markdown("### Session History - Analyses & Conversations")

                with gr.Row():
                    with gr.Column():
                        view_mode = gr.Radio(
                            choices=["📊 Analyses Only", "💬 Full Conversation", "📋 Everything"],
                            value="📋 Everything",
                            label="View Mode"
                        )
                        refresh_history_btn = gr.Button("🔄 Refresh History", variant="secondary", size="lg")

                history_display = gr.Markdown("*Click refresh to load history...*")

                def show_history(session_state, view_mode):
                    if 'session_id' not in session_state:
                        return "No history yet. Run an analysis or start chatting!"

                    session_id = session_state['session_id']

                    analyses = self.history_manager.get_recent_analyses(session_id, limit=10)

                    conversations = self.history_manager.get_session_history(session_id, limit=50)

                    if not analyses and not conversations:
                        return "No history yet. Run an analysis or start chatting!"

                    history_md = f"# 📜 Session History\n\n**Session ID:** `{session_id[:8]}...`\n\n"

                    if view_mode == "📊 Analyses Only":
                        if not analyses:
                            return history_md + "\n*No analyses yet. Run a stock analysis first!*"

                        history_md += "## 📊 Stock Analyses\n\n"
                        for i, a in enumerate(analyses, 1):
                            decision_emoji = "🟢" if a['decision'] == "BUY" else "🔴" if a['decision'] == "SELL" else "🟡"
                            rec = a.get('recommendation', {})

                            history_md += f"""
---
### {i}. {decision_emoji} **{a['ticker']}** - {a['decision']} ({a['confidence']}% confidence)
**📅 Time:** {a['timestamp']}
**🎯 Primary Driver:** {rec.get('primary_driver', 'N/A')}
**💰 Entry Price:** ${rec.get('entry_price', 'N/A')}
**🛑 Stop Loss:** ${rec.get('stop_loss', 'N/A')}

**📝 Quick Summary:**
{rec.get('decision_rationale', 'N/A')[:200]}...

"""

                    elif view_mode == "💬 Full Conversation":
                        if not conversations:
                            return history_md + "\n*No conversations yet. Start chatting!*"

                        history_md += "## 💬 Conversation History\n\n"
                        for i, (role, content) in enumerate(conversations, 1):
                            if role == "user":
                                history_md += f"\n**👤 User ({i}):**\n> {content}\n"
                            else:
                                history_md += f"\n**🤖 Assistant:**\n{content[:500]}{'...' if len(content) > 500 else ''}\n"

                    else:
                        if analyses:
                            history_md += "## 📊 Stock Analyses\n\n"
                            for i, a in enumerate(analyses, 1):
                                decision_emoji = "🟢" if a['decision'] == "BUY" else "🔴" if a['decision'] == "SELL" else "🟡"
                                rec = a.get('recommendation', {})

                                history_md += f"""
### {i}. {decision_emoji} **{a['ticker']}** - {a['decision']} ({a['confidence']}%)
**📅 {a['timestamp']}** | **🎯 {rec.get('primary_driver', 'N/A')}** | **💰 Entry: ${rec.get('entry_price', 'N/A')}**

"""

                        if conversations:
                            history_md += "\n---\n\n## 💬 Recent Conversations\n\n"

                            conv_count = len(conversations)
                            history_md += f"**Total Messages:** {conv_count}\n\n"

                            recent_convs = conversations[-20:] if len(conversations) > 20 else conversations

                            for i, (role, content) in enumerate(recent_convs, 1):
                                if role == "user":
                                    history_md += f"\n**👤 User:**\n> {content}\n"
                                else:
                                    truncated = content[:300] + "..." if len(content) > 300 else content
                                    history_md += f"\n**🤖 Assistant:**\n{truncated}\n"

                            if len(conversations) > 20:
                                history_md += f"\n\n*Showing last 20 of {conv_count} messages*\n"

                    return history_md

                refresh_history_btn.click(
                    fn=show_history,
                    inputs=[session_state, view_mode],
                    outputs=[history_display]
                )

                view_mode.change(
                    fn=show_history,
                    inputs=[session_state, view_mode],
                    outputs=[history_display]
                )

            analyze_btn.click(
                fn=self.analyze_stock_ui,
                inputs=[ticker_input, risk_input, session_state],
                outputs=[report_output, chat_status, session_state]
            )

            send_btn.click(
                fn=self.chat_response,
                inputs=[chat_input, chatbot, session_state],
                outputs=[chatbot, session_state]
            ).then(
                lambda: "",
                outputs=[chat_input]
            )

            chat_input.submit(
                fn=self.chat_response,
                inputs=[chat_input, chatbot, session_state],
                outputs=[chatbot, session_state]
            ).then(
                lambda: "",
                outputs=[chat_input]
            )

        return demo


In [27]:
# Launch UI function

def launch_ui(llm_provider: str = None, share: bool = True):
    """Launch the Gradio interface (Colab-compatible)"""
    ui = TradingAgentUI(llm_provider=llm_provider)
    demo = ui.create_interface()

    demo.launch(
        share=share,
        debug=True,
        inline=False
    )


In [28]:
RUN_WITH_UI = True
# RUN_WITH_UI = False


In [ ]:
if __name__ == "__main__":
    if RUN_WITH_UI:
        print("Launching Gradio UI...")
        launch_ui(share=True)
    else:
        ticker = "NVDA"

        print(f"\nAnalyzing {ticker}...")
        print("Primary: Sentiment (AskNews, Twitter, Reddit, News)")
        print("Confirmation: Technical Indicators (RSI, Alligator, Squeeze)")

        try:
            result = analyze_stock(
                ticker=ticker,
                risk_tolerance="medium",
                save_to_file=True
            )

            print_analysis_report(result)

            rec = result.get('recommendation', {})
            if rec.get('decision'):
                print(f"\nFinal: {rec['decision']} ({rec.get('confidence', 0)}% confidence)")
                print(f"Driver: {rec.get('primary_driver', 'N/A')}")

        except Exception as e:
            print(f"\nError: {str(e)}")
            import traceback
            traceback.print_exc()

Launching Gradio UI...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://74261745d02ff53b3e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Using LLM Provider: OLLAMA
   Model: qwen3:latest
   Base URL: http://localhost:11434

Starting multi-agent analysis for NVDA...
ANALYSIS HIERARCHY:
1- PRIMARY: Sentiment (News, Twitter, Reddit)
2- CONFIRMATION: Technical Indicators (RSI, Alligator, Squeeze)
3- SUPPORTING: Fundamental Analysis
4- CONSTRAINT: Risk Management
📊 Fetching data for NVDA from yfinance...
✅ Using yfinance data: 251 days

Analysis saved to: NVDA_analysis_ollama_20260126_223357.json
